# Climate Risk Disclosure Scrapping
This webscrapping tool is developed to download data from the Climate Reporting Entity (CRE) Search Hub. 

```
In recognition of the ongoing impact that climate change has on New Zealand the Government introduced a requirement for large entities to prepare and lodge annual climate-related disclosures (climate statements). These entities are called Climate Reporting Entities (CREs) under the Financial Markets Conduct Act 2013.  

The aim of requiring CREs to consider and report on climate-related risks and opportunities is to encourage a transition to a low-emissions future.

CREs are required to lodge climate statements or exemption notices with the Registrar of Financial Service Providers within 4 months after the balance date of the entity. CREs with a balance date of 31 December 2023 will be the first to file their climate statements which are due by 30 April 2024.

Which entities are Climate Reporting Entities (CREs)
Around 200 entities in New Zealand will be required to prepare and lodge climate statements or exemption notices. CREs are a subset of FMC reporting entities, and include: 
 - Large registered banks, credit unions, and building societies. Those with total assets of more than $1 billion. 
 - Large managers of registered investment schemes (other than restricted schemes). Those with greater than $1 billion in total assets under management. 
 - Large licensed insurers. Those with greater than $1 billion in total assets or annual gross premium income greater than $250 million. 
 - Large listed issuers of quoted equity securities or quoted debt securities. An equity issuer is large if the market price or fair value of all of its equity securities exceeds $60 million and a debt issuer is large if the face value of its quoted debt exceeds $60 million. Issuers listed on growth markets are excluded from the climate reporting entity definition. 
 
The thresholds for each entity are calculated as at the balance date of their 2 preceding accounting periods. These thresholds will be amended from time to time to reflect the movements in the consumers price index.
```
The CRE search hub is this link: https://crd-app.companiesoffice.govt.nz/dashboard/

```
From 1 January 2024 CREs are required to prepare climate statements and lodge them on this register. To begin with, the register only includes CREs with a reporting period up to June 2024. We'll add the remaining CREs in due course.

```




In [1]:
!pip install selenium==3.141.0
!pip install bs4
!pip install webdriver_manager

In [9]:
## import the necessary packages which we are going to use here. 
import selenium
from selenium import webdriver 
from selenium.webdriver.chrome.service import Service as ChromeService 
from webdriver_manager.chrome import ChromeDriverManager 
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import os
import pandas as pd 
import numpy as np
from bs4 import BeautifulSoup
import ast
import time 
from datetime import date


In [5]:
selenium.__version__

'3.141.0'

In [3]:
## Edit accordingly to the file path where an instance of the classes are existing
os.chdir(path=r'C:\Users\QuyenN\Offline work\1.GNS\5-Climate risk disclosure\Classes')
import WebScrapeClass
import TableBuilderClass
import TableTidyandSaveClass

In [14]:
folder = 'C:/Users/QuyenN/Offline work/1.GNS/5-Climate risk disclosure/Classes/'
# Define Chrome options
options = webdriver.ChromeOptions() 
options.headless = True 
options.add_argument("--start-maximized")
options.add_argument("--headless")  # Uncomment this line if you want to run Chrome in headless mode

# Specify the path to the Chrome driver executable
#cService = ChromeService(executable_path= "C:/Users/QuyenN/Offline work/1.GNS/5-Climate risk disclosure/Classes/chromedriver.exe")
driver = webdriver.Chrome(executable_path= "C:/Users/QuyenN/Offline work/1.GNS/5-Climate risk disclosure/Classes/chromedriver.exe")

The first step is to download the list of all entities 

In [7]:
## This URL represents the link to CRD hub.  
link= 'https://crd-app.companiesoffice.govt.nz/dashboard/'
# open the browser to activate the process using the URL defined 
driver.get(link)
# extract  data 
table_final = pd.DataFrame()
# Loop through pages 1 to 16
for page_num in range(1, 17):
    # Scroll down to reveal hidden pagination buttons
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    
    # Find the button with data-page attribute
    button_with_data_page = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, f'a[data-page="{page_num}"]'))
    )
    # Click the button
    button_with_data_page.click()
    
    # Wait for the table to become visible
    table = WebDriverWait(driver, 10).until(
        EC.visibility_of_element_located((By.CSS_SELECTOR, '#mainContent > div.entitylist > div > div.view-grid.table-responsive.has-pagination > table'))
    )

    # Do something with the table (for demonstration, print its HTML)
    print(f"Table on Page {page_num}:")
    print(table.text)
    
    # Get the table HTML
    table_html = table.get_attribute("outerHTML")
    
    
    #Extract table data
    # Parse the HTML table content with BeautifulSoup
    soup = BeautifulSoup(table_html, 'html.parser')

    # Extract table rows
    rows = soup.find_all('tr')

    # Initialize lists to store table data and associated links
    table_data = []

    # Loop through table rows
    for row in rows:
        # Extract table cells (columns)
        cells = row.find_all(['th', 'td'])
        row_data = []
        for cell in cells:
            # Extract text from cell
            cell_text = cell.get_text(strip=True)
            row_data.append(cell_text)
        links = []
        # Check if cell contains a link
        link = cell.find('a')
        if link:
            # Extract link URL
            link_url = link.get('href')
            links.append(link_url)

        # Remove duplicates using set
        unique_href_values = set(links)

        # Filter out only the elements starting with '/dashboard/reportingentity/?id='
        filtered_href_values = [href for href in unique_href_values if href.startswith('/dashboard/reportingentity/?id=')]

        row_data.append(filtered_href_values)
        # Append row data to table data
        table_data.append(row_data)

    # Convert table data to DataFrame
    table_df = pd.DataFrame(table_data)

    # After obtaining the table, you can perform any further actions required
    table_final = pd.concat([table_final, table_df], ignore_index=True)

Table on Page 1:
CRE Name
. sort ascending
Reporting Entity Status
. sort descending
NZBN
. sort descending
Actions
AA INSURANCE LIMITED Registered 9429040865966
AFT PHARMACEUTICALS LIMITED Registered 9429038010415
AIA NEW ZEALAND LIMITED Registered 9429039580948
AIR NEW ZEALAND LIMITED Registered 9429040402543
AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED Registered 9429038259210
ANZ INVESTMENT SERVICES (NEW ZEALAND) LIMITED Registered 9429039496881
ANZ NEW ZEALAND INVESTMENTS LIMITED Registered 9429039355140
ARBORGEN HOLDINGS LIMITED Registered 9429037034740
ARGOSY PROPERTY LIMITED Registered 9429030890039
ARVIDA GROUP LIMITED Registered 9429041061350
Table on Page 2:
CRE Name
. sort ascending
Reporting Entity Status
. sort descending
NZBN
. sort descending
Actions
ASB BANK LIMITED Registered 9429039435743
ASB GROUP INVESTMENTS LIMITED Registered 9429039025791
ASSET PLUS LIMITED Registered 9429031430098
ASTERON LIFE LIMITED Registered 9429040905464
AUCKLAND COUNCIL Registered 94290000347

Table on Page 14:
CRE Name
. sort ascending
Reporting Entity Status
. sort descending
NZBN
. sort descending
Actions
SOUTHERN CROSS MEDICAL CARE SOCIETY Registered 9429043036400
SOUTHLAND BUILDING SOCIETY Registered 9429043042654
SPARK FINANCE LIMITED Registered 9429039076847
SPARK NEW ZEALAND LIMITED Registered 9429039661098
STEEL & TUBE HOLDINGS LIMITED Registered 9429040949390
STRIDE INVESTMENT MANAGEMENT LIMITED Registered 9429042189435
STRIDE PROPERTY LIMITED Registered 9429035963950
SUMMERSET GROUP HOLDINGS LIMITED Registered 9429035141679
SWISS RE LIFE & HEALTH AUSTRALIA LIMITED Registered 9429040975696
T&G GLOBAL LIMITED Registered 9429040750705
Table on Page 15:
CRE Name
. sort ascending
Reporting Entity Status
. sort descending
NZBN
. sort descending
Actions
TEMPLETON EMERGING MARKETS INVESTMENT TRUST PUBLIC LIMITED COMPANY Registered 9429038745980
THE A2 MILK COMPANY LIMITED Registered 9429037368845
THE CITY OF LONDON INVESTMENT TRUST PLC Registered 9429036466658
THE COLONIA

In [8]:
table_final.columns = ['CRE Name','Reporting Entity Status','NZBN','Actions','Link']
table_final.drop(0, inplace=True)
table_final.head(5)

,CRE Name,Reporting Entity Status,NZBN,Actions,Link
1,AA INSURANCE LIMITED,Registered,9429040865966,View details,[/dashboard/reportingentity/?id=8b6b5ca5-8394-...
2,AFT PHARMACEUTICALS LIMITED,Registered,9429038010415,View details,[/dashboard/reportingentity/?id=a96b5ca5-8394-...
3,AIA NEW ZEALAND LIMITED,Registered,9429039580948,View details,[/dashboard/reportingentity/?id=8d6b5ca5-8394-...
4,AIR NEW ZEALAND LIMITED,Registered,9429040402543,View details,[/dashboard/reportingentity/?id=ab6b5ca5-8394-...
5,AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED,Registered,9429038259210,View details,[/dashboard/reportingentity/?id=696c5ca5-8394-...


In [16]:
today = date.today().strftime("%d/%m/%Y")
table_final.to_csv(f'{folder}/table_final.csv')

In [17]:
today

'07/05/2024'

In [20]:
table_final = pd.read_csv(f'{folder}/table_final.csv')
## This URL represents the link to CRD hub.  
masterlink= 'https://crd-app.companiesoffice.govt.nz'

links_final = pd.DataFrame()
for index, row in table_final.iterrows():
    firm_name = row['CRE Name']
    links = [masterlink + link for link in ast.literal_eval(row.Link)]
    df = pd.DataFrame({'CRE Name': [row['CRE Name']] * len(links), 'Link': links})
    links_final = pd.concat([links_final, df],ignore_index = True)
links_final.head(5)
links_final.to_csv(f'{folder}/links_final.csv')

This step is to extract the link and download pdf files from all  entities again 

In [21]:
links_final = pd.read_csv(f'{folder}/links_final.csv')
## This URL represents the link to CRD hub.  
masterlink= 'https://crd-app.companiesoffice.govt.nz'

# extract  data 
table_details_final = pd.DataFrame()

for index, row in links_final.iterrows():
    firm_name = row['CRE Name']
    print(firm_name)
    link = row.Link

    # open the browser to activate the process using the URL defined 
    driver.get(link)
    
    time.sleep(3)
       
    # Wait for the table to become visible
    table = WebDriverWait(driver, 30).until(
        EC.visibility_of_element_located((By.CSS_SELECTOR, "table"))
    )
    
    text = table.text
    print(text)
    # Create a dictionary with the row data
    row_data = pd.DataFrame([{
        'Link': link,
        'Company Name': firm_name,
        'Text': text
    }])
    # After obtaining the table, you can perform any further actions required
    table_details_final = pd.concat([table_details_final, row_data], ignore_index=True)

table_details_final.to_csv(f'{folder}/ltable_details_final.csv') 

AA INSURANCE LIMITED
Reporting Period
. sort descending
Actions
There are no records to display.
AFT PHARMACEUTICALS LIMITED
Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
AIA NEW ZEALAND LIMITED
Reporting Period
. sort descending
Actions
31-12-2023 - Click to view Climate Statement/s
AIR NEW ZEALAND LIMITED
Reporting Period
. sort descending
Actions
There are no records to display.
AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
Scheme
. sort ascending
Scheme Status
. sort descending
Reporting Entity
. sort descending
Actions
AMP INVESTMENT TRUST (SCH10721) Registered AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
AMP KIWISAVER SCHEME (SCH10367) Registered AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
AMP MANAGED FUNDS (SCH13265) Registered AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
AMP PASSIVE PERSONAL RETIREMENT PLAN - INTERNATIONAL PASSIVE SHARES INVESTMENT FUND (SCH10821) Registered AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
AMP PASSIVE PERSONAL RETIREMENT PLAN - NEW ZEA

Reporting Period
. sort descending
Actions
There are no records to display.
CHRISTCHURCH CITY HOLDINGS LIMITED
Reporting Period
. sort descending
Actions
There are no records to display.
CHRISTCHURCH INTERNATIONAL AIRPORT LIMITED
Reporting Period
. sort descending
Actions
There are no records to display.
CHUBB LIFE INSURANCE NEW ZEALAND LIMITED
Reporting Period
. sort descending
Actions
31-12-2023 - Click to view Climate Statement/s
CITIBANK, N.A.
Reporting Period
. sort descending
Actions
31-12-2023 - Click to view Climate Statement/s
COMMONWEALTH BANK OF AUSTRALIA
Reporting Period
. sort descending
Actions
There are no records to display.
COMVITA LIMITED
Reporting Period
. sort descending
Actions
There are no records to display.
CONTACT ENERGY LIMITED
Reporting Period
. sort descending
Actions
There are no records to display.
COOPERATIEVE RABOBANK U.A.
Reporting Period
. sort descending
Actions
31-12-2023 - Click to view Climate Statement/s
DELEGAT GROUP LIMITED
Reporting Period
. so

Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
MANAWA ENERGY LIMITED
Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
MARLIN GLOBAL LIMITED
Reporting Period
. sort descending
Actions
There are no records to display.
MARSDEN MARITIME HOLDINGS LIMITED
Reporting Period
. sort descending
Actions
There are no records to display.
MEDICAL FUNDS MANAGEMENT LIMITED
Scheme
. sort ascending
Scheme Status
. sort descending
Reporting Entity
. sort descending
Actions
MAS INVESTMENT FUNDS  (SCH13659) Registered MEDICAL FUNDS MANAGEMENT LIMITED
MAS KIWISAVER SCHEME (SCH10705) Registered MEDICAL FUNDS MANAGEMENT LIMITED
MAS RETIREMENT SAVINGS SCHEME (SCH10706) Registered MEDICAL FUNDS MANAGEMENT LIMITED
MERCER (N.Z.) LIMITED
Scheme
. sort ascending
Scheme Status
. sort descending
Reporting Entity
. sort descending
Actions
DEFENCE FORCE SUPERANNUATION SCHEME (SCH11069) Registered MERCER (N.Z.) LIMITED
MERCER FLEXISAVER (SCH10280) Registered MERCER (N.Z

Reporting Period
. sort descending
Actions
There are no records to display.
SOUTHLAND BUILDING SOCIETY
Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
SPARK FINANCE LIMITED
Reporting Period
. sort descending
Actions
There are no records to display.
SPARK NEW ZEALAND LIMITED
Reporting Period
. sort descending
Actions
There are no records to display.
STEEL & TUBE HOLDINGS LIMITED
Reporting Period
. sort descending
Actions
There are no records to display.
STRIDE INVESTMENT MANAGEMENT LIMITED
Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
STRIDE PROPERTY LIMITED
Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
SUMMERSET GROUP HOLDINGS LIMITED
Reporting Period
. sort descending
Actions
31-12-2023 - Click to view Climate Statement/s
SWISS RE LIFE & HEALTH AUSTRALIA LIMITED
Reporting Period
. sort descending
Actions
31-12-2023 - Click to view Climate Statement/s
T&G GLOBAL LIMITED
Reporting Period
. sort descending


In [25]:
#Work a bit more on table detail
table_details_final = pd.read_csv(f'{folder}/table_details_final.csv')
# Clean up the 'Text' column
table_details_final['text_new'] = table_details_final['Text'].str.replace('\n', '|') \
                                                              .str.replace('Reporting Period', '') \
                                                              .str.replace('. sort descending', '') \
                                                              .str.replace('. sort ascending', '') \
                                                              .str.replace('Scheme', '')\
                                                              .str.replace('Scheme Status', '')\
                                                              .str.replace('Reporting Entity', '')\
                                                              .str.replace('Status', '')\
                                                              .str.replace('Registered', '_')\
                                                   .str.replace('Actions', '')\


# Display the DataFrame
print(table_details_final.head(5))

table_details_final.to_csv(f'{folder}/table_details_final.new.csv') 

   Unnamed: 0                                               Link  \
0           0  https://crd-app.companiesoffice.govt.nz/dashbo...   
1           1  https://crd-app.companiesoffice.govt.nz/dashbo...   
2           2  https://crd-app.companiesoffice.govt.nz/dashbo...   
3           3  https://crd-app.companiesoffice.govt.nz/dashbo...   
4           4  https://crd-app.companiesoffice.govt.nz/dashbo...   

                                Company Name  \
0                       AA INSURANCE LIMITED   
1                AFT PHARMACEUTICALS LIMITED   
2                    AIA NEW ZEALAND LIMITED   
3                    AIR NEW ZEALAND LIMITED   
4  AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED   

                                                Text  \
0  Reporting Period\n. sort descending\nActions\n...   
1  Reporting Period\n. sort descending\nActions\n...   
2  Reporting Period\n. sort descending\nActions\n...   
3  Reporting Period\n. sort descending\nActions\n...   
4  Scheme\n. sort asce

In [26]:
table_details_final = pd.read_csv(f'{folder}/table_details_final.new.csv') 

#Get the scheme list:
# Search for rows containing 'scheme' in the 'Text' column
scheme_rows = table_details_final[table_details_final['Text'].str.contains('Scheme', case=False)]

print(scheme_rows.shape)
# Display the rows containing 'scheme'
scheme_rows.head(5)


## This URL represents the link to CRD hub.  
masterlink= 'https://crd-app.companiesoffice.govt.nz'

# extract  data 
table_details_final_scheme = pd.DataFrame()

for index, row in scheme_rows.iterrows():
    firm_name = row['Company Name']
    print(firm_name)
    link = row.Link

    # open the browser to activate the process using the URL defined 
    driver.get(link)
    
    time.sleep(3)
       
    # Wait for the table to become visible
    table = WebDriverWait(driver, 30).until(
        EC.visibility_of_element_located((By.CSS_SELECTOR, "table"))
    )
    
    text = table.text
    print(text)
    
    # Get the table HTML
    table_html = table.get_attribute("outerHTML")
    
    #Extract table data
    # Parse the HTML table content with BeautifulSoup
    soup = BeautifulSoup(table_html, 'html.parser')

    # Extract table rows
    rows = soup.find_all('tr')

    # Initialize lists to store table data and associated links
    table_data = []

    # Loop through table rows
    for row in rows[2:]:
        # Extract table cells (columns)
        cells = row.find_all(['td'])
        row_data = []
        for cell in cells:
            # Extract text from cell
            cell_text = cell.get_text(strip=True)
            row_data.append(cell_text)
        links = []
        # Check if cell contains a link
        link = cell.find('a')
        if link:
            # Extract link URL
            link_url = link.get('href')
            links.append(link_url)

        # Remove duplicates using set
        unique_href_values = set(links)

        # Filter out only the elements starting with '/dashboard/reportingentity/?id='
        filtered_href_values = [href for href in unique_href_values if href.startswith('/dashboard/reportingentity/')]

        row_data.append(filtered_href_values)
        # Append row data to table data
        table_data.append(row_data)

    # Convert table data to DataFrame
    table_df = pd.DataFrame(table_data)
    
    # After obtaining the table, you can perform any further actions required
    table_details_final_scheme = pd.concat([table_details_final_scheme, table_df], ignore_index=True)
table_details_final_scheme.columns = ['Scheme','Scheme Status','Company Name','Detail','Link']
table_details_final_scheme.dropna(subset=['Detail'],inplace=True)
table_details_final_scheme.to_csv(f'{folder}/table_details_final_scheme.csv') 

(25, 6)
AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
Scheme
. sort ascending
Scheme Status
. sort descending
Reporting Entity
. sort descending
Actions
AMP INVESTMENT TRUST (SCH10721) Registered AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
AMP KIWISAVER SCHEME (SCH10367) Registered AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
AMP MANAGED FUNDS (SCH13265) Registered AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
AMP PASSIVE PERSONAL RETIREMENT PLAN - INTERNATIONAL PASSIVE SHARES INVESTMENT FUND (SCH10821) Registered AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
AMP PASSIVE PERSONAL RETIREMENT PLAN - NEW ZEALAND PASSIVE SHARES INVESTMENT FUND (SCH10822) Registered AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
AMP PERSONAL RETIREMENT PLAN (SCH10820) Registered AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
NEW ZEALAND RETIREMENT TRUST (SCH10911) Registered AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
ANZ INVESTMENT SERVICES (NEW ZEALAND) LIMITED
Scheme
. sort ascending
Scheme Status
. sort descending
Reporting Entity


Scheme
. sort ascending
Scheme Status
. sort descending
Reporting Entity
. sort descending
Actions
GENERATE KIWISAVER SCHEME (SCH10791) Registered GENERATE INVESTMENT MANAGEMENT LIMITED
GENERATE UNIT TRUST SCHEME (SCH12736) Registered GENERATE INVESTMENT MANAGEMENT LIMITED
GOODMAN (NZ) LIMITED
Scheme
. sort ascending
Scheme Status
. sort descending
Reporting Entity
. sort descending
Actions
GOODMAN PROPERTY TRUST (SCH11225) Registered GOODMAN (NZ) LIMITED
HARBOUR ASSET MANAGEMENT LIMITED
Scheme
. sort ascending
Scheme Status
. sort descending
Reporting Entity
. sort descending
Actions
HARBOUR INVESTMENT FUNDS (SCH10815) Registered HARBOUR ASSET MANAGEMENT LIMITED
HUNTER INVESTMENT FUNDS (SCH11176) Registered HARBOUR ASSET MANAGEMENT LIMITED
KIWI WEALTH INVESTMENTS LIMITED PARTNERSHIP
Scheme
. sort ascending
Scheme Status
. sort descending
Reporting Entity
. sort descending
Actions
KIWI WEALTH SUPER SCHEME (SCH10814) Registered KIWI WEALTH INVESTMENTS LIMITED PARTNERSHIP
KIWI WEALTH LIM

In [28]:
table_final = pd.read_csv(f'{folder}/table_details_final_scheme.csv') 
links_scheme_final = pd.DataFrame()
for index, row in table_final.iterrows():
    firm_name = row['Company Name']
    links = [masterlink + link for link in ast.literal_eval(row.Link)]
    # Create a list of dictionaries containing the data
    data = [{'Scheme Name': row['Scheme'], 'CRE Name': row['Company Name'], 'Link': link} for link in links]
    # Create the DataFrame
    df = pd.DataFrame(data)    
    links_scheme_final = pd.concat([links_scheme_final, df],ignore_index = True)
links_scheme_final.head(5)
links_scheme_final.to_csv(f'{folder}/links_scheme_final.csv')

links_scheme_final = pd.read_csv(f'{folder}/links_scheme_final.csv')
# extract  data 
table_details_final_scheme_detail = pd.DataFrame()

for index, row in links_scheme_final.iterrows():
    firm_name = row['CRE Name']
    print(firm_name)
    link = row.Link

    # open the browser to activate the process using the URL defined 
    driver.get(link)
    
    time.sleep(3)
       
    # Wait for the table to become visible
    table = WebDriverWait(driver, 30).until(
        EC.visibility_of_element_located((By.CSS_SELECTOR, "table"))
    )
    
    text = table.text
    print(text)
    # Create a dictionary with the row data
    row_data = pd.DataFrame([{
        'Link': link,
        'Company Name': firm_name,
        'Text': text
    }])
    # After obtaining the table, you can perform any further actions required
    table_details_final_scheme_detail = pd.concat([table_details_final_scheme_detail, row_data], ignore_index=True)

table_details_final_scheme_detail.to_csv(f'{folder}/table_details_final.scheme_detail.csv') 


AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED
Reporting Period
. sort descending
Actions
There are no records to display.
ANZ INVESTMENT SERVICES (NEW ZEALAND) LIMITED
Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
ANZ NEW ZEALAND INVESTMENTS LIMITED
Reporting Period
. sort descen

Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
FUNDROCK NZ LIMITED
Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
FUNDROCK NZ LIMITED
Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
FUNDROCK NZ LIMITED
Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
FUNDROCK NZ LIMITED
Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
FUNDROCK NZ LIMITED
Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
FUNDROCK NZ LIMITED
Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
GENERATE INVESTMENT MANAGEMENT LIMITED
Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
GENERATE INVESTMENT MANAGEMENT LIMITED
Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
GOODMAN (NZ) LIMITED
Reporting Period
. sort descending
Actions
31-03-2024 - Due by 31-07-2024
HARBOUR ASSET MANAGEMENT LIMITED
Reportin

In [29]:
links_scheme_final =pd.read_csv(f'{folder}/links_scheme_final.csv')

# Clean up the 'Text' column
table_details_final_scheme_detail['text_new'] = table_details_final_scheme_detail['Text'].str.replace('\n', '|') \
                                                              .str.replace('Reporting Period', '') \
                                                              .str.replace('. sort descending', '') \
                                                              .str.replace('. sort ascending', '') \
                                                              .str.replace('Scheme', '')\
                                                              .str.replace('Scheme Status', '')\
                                                              .str.replace('Reporting Entity', '')\
                                                              .str.replace('Status', '')\
                                                              .str.replace('Registered', '_')\
                                                   .str.replace('Actions', '')\



table_details_final_scheme_detail = pd.merge(table_details_final_scheme_detail, 
                                            links_scheme_final[['Link','Scheme Name']],
                                            how = 'left',
                                            left_on = 'Link',
                                            right_on = 'Link')
# Display the DataFrame
print(table_details_final_scheme_detail.head(5))
table_details_final_scheme_detail.to_csv(f'{folder}/table_details_final.scheme_detail.new.csv') 

                                                Link  \
0  https://crd-app.companiesoffice.govt.nz/dashbo...   
1  https://crd-app.companiesoffice.govt.nz/dashbo...   
2  https://crd-app.companiesoffice.govt.nz/dashbo...   
3  https://crd-app.companiesoffice.govt.nz/dashbo...   
4  https://crd-app.companiesoffice.govt.nz/dashbo...   

                                Company Name  \
0  AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED   
1  AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED   
2  AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED   
3  AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED   
4  AMP WEALTH MANAGEMENT NEW ZEALAND LIMITED   

                                                Text  \
0  Reporting Period\n. sort descending\nActions\n...   
1  Reporting Period\n. sort descending\nActions\n...   
2  Reporting Period\n. sort descending\nActions\n...   
3  Reporting Period\n. sort descending\nActions\n...   
4  Reporting Period\n. sort descending\nActions\n...   

                            text_new 

In [30]:
table_details_final =pd.read_csv(f'{folder}/table_details_final.new.csv') 
table_details_final_scheme_detail =pd.read_csv(f'{folder}/table_details_final.scheme_detail.new.csv') 
table_details_final_noscheme = table_details_final[table_details_final['Text'].str.contains('Scheme', case=False) == False]
table_details_final_noscheme = pd.concat([table_details_final_noscheme,
                                          table_details_final_scheme_detail],ignore_index = True)
table_details_final_noscheme.drop(columns =['Unnamed: 0.1','Unnamed: 0'],inplace=True)
table_details_final_noscheme.head(5)
table_details_final_noscheme['text_new'] = table_details_final_noscheme['text_new'].str.replace('|', '') 
# Set 'status' column based on conditions
table_details_final_noscheme['status'] = np.where(
    table_details_final_noscheme['text_new'].str.contains('no records', case=False),
    'No records',
    ''
)

table_details_final_noscheme['status'] = np.where(
    table_details_final_noscheme['text_new'].str.contains('Due ', case=False),
    'Have records but not submitted',
    table_details_final_noscheme['status']
)


table_details_final_noscheme['status'] = np.where(
    table_details_final_noscheme['text_new'].str.contains('Click to view Climate Statement', case=False),
    'Have records and have submitted',
    table_details_final_noscheme['status']
)


print(table_details_final_noscheme['status'].value_counts())

# Set 'status' column based on scheme 
table_details_final_noscheme['scheme'] = np.where(
    table_details_final_noscheme['Scheme Name'].isnull()==True,
    'Company',
    'Investment Scheme'
)

print(table_details_final_noscheme['scheme'].value_counts())
# Custom function to split date strings
def split_dates(date_str):
    if " - Due by " in date_str:
        due_date, submit_date = date_str.split(" - Due by ")
        return due_date, submit_date
    else:
        return "0", "0"

# Apply the function to each row of the DataFrame
table_details_final_noscheme[['Due Date', 'Submit Date']] = table_details_final_noscheme['text_new'].apply(lambda x: pd.Series(split_dates(x)))

# Print the modified DataFrame
print(table_details_final_noscheme.head(5))
table_details_final_noscheme.to_csv(f'{folder}/table_details_final_noscheme.final.csv') 


status
Have records but not submitted     151
No records                          77
Have records and have submitted     30
Name: count, dtype: int64
scheme
Company              135
Investment Scheme    123
Name: count, dtype: int64
                                                Link  \
0  https://crd-app.companiesoffice.govt.nz/dashbo...   
1  https://crd-app.companiesoffice.govt.nz/dashbo...   
2  https://crd-app.companiesoffice.govt.nz/dashbo...   
3  https://crd-app.companiesoffice.govt.nz/dashbo...   
4  https://crd-app.companiesoffice.govt.nz/dashbo...   

                  Company Name  \
0         AA INSURANCE LIMITED   
1  AFT PHARMACEUTICALS LIMITED   
2      AIA NEW ZEALAND LIMITED   
3      AIR NEW ZEALAND LIMITED   
4    ARBORGEN HOLDINGS LIMITED   

                                                Text  \
0  Reporting Period\n. sort descending\nActions\n...   
1  Reporting Period\n. sort descending\nActions\n...   
2  Reporting Period\n. sort descending\nActions\n...   
3

In [31]:
table_details_final = pd.read_csv(f'{folder}/table_details_final_noscheme.final.csv') 

#Get the disclose list:
# Search for rows containing 'scheme' in the 'Text' column
disclose_rows = table_details_final[table_details_final['status'] =='Have records and have submitted']

print(disclose_rows.shape)
# Display the rows containing 'scheme'
disclose_rows.head(5)


(30, 10)


,Unnamed: 0,Link,Company Name,Text,text_new,Scheme Name,status,scheme,Due Date,Submit Date
2,2,https://crd-app.companiesoffice.govt.nz/dashbo...,AIA NEW ZEALAND LIMITED,Reporting Period\n. sort descending\nActions\n...,31-12-2023 - Click to view Climate Statement/s,NaN,Have records and have submitted,Company,0,0
13,13,https://crd-app.companiesoffice.govt.nz/dashbo...,BANK OF CHINA (NEW ZEALAND) LIMITED,Reporting Period\n. sort descending\nActions\n...,31-12-2023 - Click to view Climate Statement/s,NaN,Have records and have submitted,Company,0,0
16,16,https://crd-app.companiesoffice.govt.nz/dashbo...,BRISCOE GROUP LIMITED,Reporting Period\n. sort descending\nActions\n...,31-01-2024 - Click to view Climate Statement/s,NaN,Have records and have submitted,Company,0,0
17,17,https://crd-app.companiesoffice.govt.nz/dashbo...,CDL INVESTMENTS NEW ZEALAND LIMITED,Reporting Period\n. sort descending\nActions\n...,31-12-2023 - Click to view Climate Statement/s,NaN,Have records and have submitted,Company,0,0
18,18,https://crd-app.companiesoffice.govt.nz/dashbo...,CHANNEL INFRASTRUCTURE NZ LIMITED,Reporting Period\n. sort descending\nActions\n...,31-12-2023 - Click to view Climate Statement/s,NaN,Have records and have submitted,Company,0,0


In [457]:
# extract  data 
for index, row in disclose_rows.head(1).iterrows():
    firm_name = row['Company Name']
    print(firm_name)
    link = row.Link

    # open the browser to activate the process using the URL defined 
    driver.get(link)
    time.sleep(3)
           
    # Wait for the table to become visible
    table = WebDriverWait(driver, 30).until(
        EC.visibility_of_element_located((By.CSS_SELECTOR, "table"))
    )
    
    text = table.text
    print(text)
    
    # Get the table HTML
    table_html = table.get_attribute("outerHTML")
    #Extract table data
    # Parse the HTML table content with BeautifulSoup
    soup = BeautifulSoup(table_html, 'html.parser')
    
    rows = soup.find_all('tr')

    # Initialize lists to store table data and associated links
    table_data = []

    # Loop through table rows
    for row in rows[1:]:
        print(row.text)
        # Extract table cells (columns)
        cells = row.find_all(['td'])
        row_data = []
        for cell in cells:
            # Extract text from cell
            cell_text = cell.get_text(strip=True)
            row_data.append(cell_text)
        links = []
        # Check if cell contains a link
        new_link = cell.find('a')
        if new_link:
            # Extract link URL
            new_link.click()
    


BRISCOE GROUP LIMITED
Reporting Period
. sort descending
Actions
31-01-2024 - Click to view Climate Statement/s

Reporting Period. sort descendingActions31-01-2024 - Click to view Climate Statement/s View detailsThere are no records to display.You don't have permissions to view these records.Error completing request.  Loading... Create×Close Edit×Close View details×Close Delete×CloseAre you sure you want to delete this record?DeleteCancel Error×CloseWe're sorry, an error has occurred.Close


Reporting Period. sort descendingActions
31-01-2024 - Click to view Climate Statement/s View details


TypeError: 'NoneType' object is not callable